# Quan sat ket qua cac module trong `src/`

Notebook nay minh hoa dau vao, dau ra va cach cac module ket noi voi nhau. De notebook chay nhanh va khong can tai ImageNet weights, model duoc tao voi `pretrained=False`.


In [1]:
from pathlib import Path
import sys

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
    
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_module import CIFAR10DataModule
from src.logger import TensorBoardLogger
from src.model_builder import TransferModelBuilder
from src.trainer import ModelTrainer

torch.manual_seed(42)
print("Project root:", ROOT)
print("PyTorch:", torch.__version__)
print("Device:", "cuda:0" if torch.cuda.is_available() else "cpu")

Project root: D:\UTH\Deeplearning\lab\Practice-2---Hands-on-practice-with-pre-trained-neural-network-architectures
PyTorch: 2.15.0+cu132
Device: cuda:0


## 1. `data_module.py`: tao dataset va DataLoader


In [2]:
data_module = CIFAR10DataModule(
    data_dir=str(ROOT / "data"),
    batch_size=4,
    num_workers=0,
    image_size=224,
)
data_module.setup()

print("Train dataset:", type(data_module.train_dataset).__name__)
print("Train samples:", len(data_module.train_dataset))
print("Test samples:", len(data_module.test_dataset))
print("Train transform:", data_module.train_transform)
print("Test transform:", data_module.test_transform)

train_images, train_labels = next(iter(data_module.train_dataloader()))
print("Train image batch shape:", train_images.shape)
print("Train label batch shape:", train_labels.shape)
print("Train image dtype:", train_images.dtype)
print("First labels:", train_labels.tolist())

Train dataset: CIFAR10
Train samples: 50000
Test samples: 10000
Train transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Test transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Train image batch shape: torch.Size([4, 3, 224, 224])
Train label batch shape: torch.Size([4])
Train image dtype: torch.float32
First labels: [6, 0, 4, 1]


## 2. `model_builder.py`: tao model va freeze backbone


In [3]:
builder = TransferModelBuilder(
    model_name="resnet18",
    num_classes=10,
    pretrained=True,
)
model = builder.build()
total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model)
print("Output classes:", model.fc.out_features)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Frozen parameters:", total_params - trainable_params)

with torch.no_grad():
    model_output = model(train_images)
print("Model output shape:", model_output.shape)
print("Predicted classes:", model_output.argmax(dim=1).tolist())

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## 3. `trainer.py`: mot buoc train va evaluate

De quan sat nhanh, phan nay dung mot tap du lieu gia lap nho co cung shape voi anh CIFAR-10 sau resize.

In [6]:
demo_images = torch.randn(8, 3, 224, 224)
demo_labels = torch.randint(0, 10, (8,))
demo_loader = DataLoader(
    TensorDataset(demo_images, demo_labels),
    batch_size=4,
    shuffle=False,
)

demo_model = TransferModelBuilder(
    model_name="resnet18",
    num_classes=10,
    pretrained=True,
).build()
trainer = ModelTrainer(
    model=demo_model,
    train_dataloader=demo_loader,
    test_dataloader=demo_loader,
    learning_rate=0.001,
    optimizer_name="Adam",
    device=torch.device("cpu"),
    weight_decay=0.0001,
)

train_loss, train_accuracy = trainer.training_loop()
test_loss, test_accuracy = trainer.evaluate()
print(f"Train loss: {train_loss:.4f}")
print(f"Train accuracy: {train_accuracy:.2f}%")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2f}%")

Train loss: 2.2155
Train accuracy: 37.50%
Test loss: 2.1271
Test accuracy: 25.00%


## 4. `logger.py`: ghi TensorBoard va checkpoint


In [7]:
logger = TensorBoardLogger(
    log_dir=ROOT / "runs",
    experiment_name="src_output_demo",
    checkpoint_dir=ROOT / "checkpoints",
)
logger.add_scalar("Demo/train_loss", train_loss, step=1)
logger.add_scalar("Demo/test_accuracy", test_accuracy, step=1)
logger.add_image_grid(
    "Demo/images",
    train_images,
    nrow=2,
    step=0,
    unnormalize=True,
)
checkpoint_path = logger.save_checkpoint(
    model=demo_model,
    optimizer=trainer.optimizer,
    epoch=1,
    metrics={"test_accuracy": test_accuracy},
    filename="src_output_demo.pth",
)
logger.flush()
logger.close()

print("TensorBoard directory:", ROOT / "runs" / "src_output_demo")
print("Checkpoint path:", checkpoint_path)

TensorBoard directory: D:\UTH\Deeplearning\lab\Practice-2---Hands-on-practice-with-pre-trained-neural-network-architectures\runs\src_output_demo
Checkpoint path: D:\UTH\Deeplearning\lab\Practice-2---Hands-on-practice-with-pre-trained-neural-network-architectures\checkpoints\src_output_demo.pth


In [ ]:
"""Thu nghiem learning rate, batch size va weight decay."""

import argparse
import copy
import csv
import random
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

from src.data_module import CIFAR10DataModule
from src.model_builder import TransferModelBuilder_ResNet18
from src.trainer import ModelTrainer


ROOT = Path(__file__).resolve().parent
SEED = 42

# Moi lan chi thay doi mot thong so so voi baseline.
CONFIGS = [
    ("baseline", 0.001, 64, 0.0001),
    ("lr_low", 0.0001, 64, 0.0001),
    ("lr_high", 0.01, 64, 0.0001),
    ("batch_small", 0.001, 32, 0.0001),
    ("batch_large", 0.001, 128, 0.0001),
    ("no_l2", 0.001, 64, 0.0),
    ("l2_high", 0.001, 64, 0.001),
]


def keep_batchnorm_eval(layer, inputs):
    """Giu BatchNorm o che do eval, khong thay doi dau vao."""
    layer.eval()
    return None


class FrozenBackboneTrainer(ModelTrainer):
    """Giu thong ke BatchNorm cua backbone co dinh."""

    def training_loop(self):
        handles = []

        try:
            for module in self.model.modules():
                if isinstance(module, torch.nn.BatchNorm2d):
                    handle = module.register_forward_pre_hook(
                        keep_batchnorm_eval
                    )
                    handles.append(handle)

            return super().training_loop()
        finally:
            for handle in handles:
                handle.remove()


def run(args):
    random.seed(SEED)
    torch.manual_seed(SEED)

    print("Dang kiem tra va nap du lieu CIFAR-10...", flush=True)

    data = CIFAR10DataModule(
        data_dir=str(ROOT / "data"),
        num_workers=0,
    )
    data.setup()

    # Tach validation tu tap train.
    # Khong dung tap test de lua chon sieu tham so.
    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(
        len(data.train_dataset),
        generator=generator,
    ).tolist()

    val_indices = indices[:5000]
    train_indices = indices[5000:]

    # Validation khong su dung phep lat anh ngau nhien.
    validation_dataset = copy.copy(data.train_dataset)
    validation_dataset.transform = data.test_transform

    if args.train_samples > 0:
        train_indices = train_indices[:args.train_samples]

    if args.val_samples > 0:
        val_indices = val_indices[:args.val_samples]

    train_set = Subset(data.train_dataset, train_indices)
    val_set = Subset(validation_dataset, val_indices)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    configs = CONFIGS if args.sweep else CONFIGS[:1]

    output_dir = ROOT / "results" / datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "experiments.csv"

    fields = [
        "experiment",
        "epoch",
        "learning_rate",
        "batch_size",
        "weight_decay",
        "train_samples",
        "val_samples",
        "train_loss",
        "train_accuracy",
        "val_loss",
        "val_accuracy",
    ]

    print(f"Device: {device}", flush=True)
    print(
        f"Train: {len(train_set)} anh | "
        f"Validation: {len(val_set)} anh",
        flush=True,
    )
    print(f"So cau hinh: {len(configs)}", flush=True)

    with output_file.open(
        "w",
        newline="",
        encoding="utf-8-sig",
    ) as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()

        for name, lr, batch_size, weight_decay in configs:
            # Khoi tao lai model va thu tu du lieu.
            random.seed(SEED)
            torch.manual_seed(SEED)

            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(SEED)

            train_loader = DataLoader(
                train_set,
                batch_size=batch_size,
                shuffle=True,
                num_workers=0,
                generator=torch.Generator().manual_seed(SEED),
            )

            val_loader = DataLoader(
                val_set,
                batch_size=batch_size,
                shuffle=False,
                num_workers=0,
            )

            print(
                f"\n{name}: lr={lr}, batch={batch_size}, "
                f"weight_decay={weight_decay}",
                flush=True,
            )
            print(
                "Dang tao ResNet18; lan dau co the can tai trong so.",
                flush=True,
            )

            model = TransferModelBuilder_ResNet18(
                num_classes=10,
                pretrained=True,
            ).build()

            trainer = FrozenBackboneTrainer(
                model=model,
                train_dataloader=train_loader,
                test_dataloader=val_loader,
                learning_rate=lr,
                optimizer_name="Adam",
                device=device,
                weight_decay=weight_decay,
            )

            for epoch in range(1, args.epochs + 1):
                print(
                    f"Dang chay epoch {epoch}/{args.epochs}...",
                    flush=True,
                )

                train_loss, train_acc = trainer.training_loop()
                val_loss, val_acc = trainer.evaluate()

                writer.writerow({
                    "experiment": name,
                    "epoch": epoch,
                    "learning_rate": lr,
                    "batch_size": batch_size,
                    "weight_decay": weight_decay,
                    "train_samples": len(train_set),
                    "val_samples": len(val_set),
                    "train_loss": train_loss,
                    "train_accuracy": train_acc,
                    "val_loss": val_loss,
                    "val_accuracy": val_acc,
                })
                file.flush()

                print(
                    f"Epoch {epoch}/{args.epochs} | "
                    f"Train acc: {train_acc:.2f}% | "
                    f"Val acc: {val_acc:.2f}% | "
                    f"Val loss: {val_loss:.4f}",
                    flush=True,
                )

            del trainer, model

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\nDa luu ket qua: {output_file}", flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--sweep", action="store_true")
    parser.add_argument("--epochs", type=int, default=1)
    parser.add_argument("--train-samples", type=int, default=256)
    parser.add_argument("--val-samples", type=int, default=128)
    args = parser.parse_args()

    if args.epochs < 1:
        parser.error("--epochs phai >= 1")

    if args.train_samples < 0 or args.val_samples < 0:
        parser.error("So anh phai >= 0; dung 0 de lay toan bo")

    run(args)

ModuleNotFoundError: No module named 'src'

Mo TensorBoard bang lenh:

```powershell
tensorboard --logdir=runs
```

Sau do mo `http://localhost:6006` de xem cac scalar va image da ghi.